# Meteorological data download: Thames Estuary

Downloads and consolidates the meteorological covariates for the dissertation model:

- Atmospheric pressure (ERA5, via Copernicus CDS)
- Wind, 10 m u/v components (ERA5, via Copernicus CDS)
- River discharge (NRFA, Thames at Kingston, station 39001)
- Catchment rainfall (NRFA, same station)

Data is pulled in small chunks so no single request trips CDS's cost limit, and so a dropped Colab session only costs the request in progress rather than the whole run. Re-running a cell skips anything already saved. For now this only covers Southend Pier, since that's the gauge you're actually modelling, extend `ACTIVE_GAUGES` later to add more once you've decided which.

Everything is written to Google Drive under `tidal_analysis_and_prediction/meteo_data/`, so it survives when the Colab runtime resets.

**Before running:** you need a free CDS account, and you need to have accepted the ERA5 licence once. See the credentials section below for the exact steps.


In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the paths
base_path = '/content/drive/MyDrive/tidal_analysis_and_prediction'
meteo_data_path = os.path.join(base_path, 'meteo_data')

# Subfolders: raw ERA5 chunks, NRFA downloads, and the final consolidated files
era5_raw_path = os.path.join(meteo_data_path, 'era5_raw')
nrfa_path = os.path.join(meteo_data_path, 'nrfa')
consolidated_path = os.path.join(meteo_data_path, 'consolidated')

for p in [meteo_data_path, era5_raw_path, nrfa_path, consolidated_path]:
    os.makedirs(p, exist_ok=True)

print(f"Project folder: {base_path}")
print(f"ERA5 raw chunks: {era5_raw_path}")
print(f"NRFA downloads: {nrfa_path}")
print(f"Consolidated output: {consolidated_path}")


Mounted at /content/drive
Project folder: /content/drive/MyDrive/tidal_analysis_and_prediction
ERA5 raw chunks: /content/drive/MyDrive/tidal_analysis_and_prediction/meteo_data/era5_raw
NRFA downloads: /content/drive/MyDrive/tidal_analysis_and_prediction/meteo_data/nrfa
Consolidated output: /content/drive/MyDrive/tidal_analysis_and_prediction/meteo_data/consolidated


In [ ]:
!pip install -q cdsapi xarray netCDF4 requests pandas numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 64.9 MB/s eta 0:00:00


## CDS credentials

Your laptop's `~/.cdsapirc` doesn't exist on Colab's virtual machine, and Colab resets its filesystem every session, so it needs recreating each time you connect to a fresh runtime. Rather than pasting your token straight into a cell (it would sit in the plain text of the .ipynb file, visible to anyone you share it with or push to GitHub), use Colab's Secrets manager:

1. Go to `cds.climate.copernicus.eu/profile` and copy your Personal Access Token.
2. In this notebook, click the key icon in the left sidebar (Secrets).
3. Add a new secret named `CDS_API_KEY` and paste the token as its value.
4. Toggle "Notebook access" on for this notebook.
5. Run the cell below. The first run will show a permission prompt, approve it.

One step has to happen on the website rather than through code: open the [ERA5 single levels dataset page](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels), scroll to the bottom of the download form, and accept the licence. Every request is rejected until this has been done once, and it only needs doing once, not every session.


In [ ]:
from google.colab import userdata
import os

CDS_URL = "https://cds.climate.copernicus.eu/api"
CDS_KEY = userdata.get('CDS_API_KEY')

config_text = f"""url: {CDS_URL}
key: {CDS_KEY}
"""

cdsapirc_path = os.path.expanduser('~/.cdsapirc')
with open(cdsapirc_path, 'w') as f:
    f.write(config_text)

print(f"Wrote credentials to {cdsapirc_path}")


Wrote credentials to /root/.cdsapirc


## Gauge coordinates and download region

Coordinates below are transcribed from the PLA national grid reference table (WGS84 lat/long column). One correction: the table lists Southend Pier's longitude as `00 43.43'N`, but N/S only applies to latitude: this is a transcription error in the source table. Given Southend sits east of Coryton (0.51°E) and west of Shivering Sands (1.11°E), the value is almost certainly `00 43.43'E`, and that's what's used below.

Hammersmith, Westminster and Herne Bay are discontinued with no coordinates in the source table, so they're left out.

Three region options are defined, from smallest to largest:
- A tight box around whichever gauges are in `ACTIVE_GAUGES`, currently just Southend Pier, since that's the one you're modelling right now.
- `GAUGE_BBOX`: a box around all 16 gauges.
- `SURGE_BBOX`: a wider southern North Sea box, covering the area where the wind and pressure that actually generate surge are found, well beyond the estuary itself.

`ACTIVE_BBOX` controls which region the download loop actually uses, and now defaults to the tight `ACTIVE_GAUGES` box. Since you'll likely only end up modelling two or three gauges, there's little point paying for the other thirteen's worth of grid points, and a smaller box gives far more headroom against CDS's cost limit. When you decide on the other gauges, add their names to `ACTIVE_GAUGES` and re-run the download cell. If they fall outside the current box you'll need to widen it and fetch again, that's a normal, expected part of working this way rather than a sign anything went wrong.


In [ ]:
def dms_to_dd(deg, minutes, hemisphere):
    """Convert degrees/decimal-minutes plus a hemisphere letter to signed decimal degrees."""
    dd = deg + minutes / 60
    if hemisphere in ('S', 'W'):
        dd *= -1
    return dd

# (degrees, minutes, hemisphere) for lat and lon, transcribed from the PLA table
GAUGES_DMS = {
    'Richmond Lock':       ((51, 27.78, 'N'), (0, 19.05, 'W')),
    'Chelsea Bridge':      ((51, 29.04, 'N'), (0, 8.98, 'W')),
    'London Bridge':       ((51, 30.45, 'N'), (0, 4.74, 'W')),
    'Charlton':            ((51, 29.66, 'N'), (0, 1.58, 'E')),
    'North Woolwich':      ((51, 29.92, 'N'), (0, 2.77, 'E')),
    'Erith':               ((51, 28.91, 'N'), (0, 11.21, 'E')),
    'Tilbury':             ((51, 27.40, 'N'), (0, 20.10, 'E')),
    'Gravesend Denton':    ((51, 26.67, 'N'), (0, 23.69, 'E')),
    'Coryton Thameshaven': ((51, 30.28, 'N'), (0, 30.31, 'E')),
    'Coryton No5 Jetty':   ((51, 30.42, 'N'), (0, 31.39, 'E')),
    'Coryton Holehaven':   ((51, 30.71, 'N'), (0, 33.07, 'E')),
    'Southend Pier':       ((51, 30.87, 'N'), (0, 43.43, 'E')),
    'Shivering Sands':     ((51, 28.37, 'N'), (1, 6.39, 'E')),
    'Margate Pile':        ((51, 23.68, 'N'), (1, 22.73, 'E')),
    'Margate Harbour':     ((51, 23.52, 'N'), (1, 22.68, 'E')),
    'Walton-on-the-Naze':  ((51, 50.60, 'N'), (1, 16.80, 'E')),
}

GAUGES = {
    name: (dms_to_dd(*lat), dms_to_dd(*lon))
    for name, (lat, lon) in GAUGES_DMS.items()
}

for name, (lat, lon) in GAUGES.items():
    print(f"{name:22s} {lat:8.4f}, {lon:8.4f}")

# Bounding box helper: returns [North, West, South, East], with a buffer in degrees
def bbox_from_gauges(gauges, buffer_deg=0.3):
    lats = [lat for lat, lon in gauges.values()]
    lons = [lon for lat, lon in gauges.values()]
    return [max(lats) + buffer_deg, min(lons) - buffer_deg,
            min(lats) - buffer_deg, max(lons) + buffer_deg]

GAUGE_BBOX = bbox_from_gauges(GAUGES, buffer_deg=0.3)
SURGE_BBOX = [55, -3, 51, 4]  # wider southern North Sea fetch region

# Which gauges you're actually modelling right now. Extend this list later, e.g.
# ACTIVE_GAUGES = ['Southend Pier', 'Tilbury', 'London Bridge']
ACTIVE_GAUGES = ['Southend Pier']
ACTIVE_GAUGES_DICT = {name: GAUGES[name] for name in ACTIVE_GAUGES}
ACTIVE_BBOX = bbox_from_gauges(ACTIVE_GAUGES_DICT, buffer_deg=0.1)

print()
print(f"GAUGE_BBOX (all 16 gauges): {GAUGE_BBOX}")
print(f"SURGE_BBOX (fetch region): {SURGE_BBOX}")
print(f"ACTIVE_GAUGES: {ACTIVE_GAUGES}")
print(f"Using ACTIVE_BBOX: {ACTIVE_BBOX}")


Richmond Lock           51.4630,  -0.3175
Chelsea Bridge          51.4840,  -0.1497
London Bridge           51.5075,  -0.0790
Charlton                51.4943,   0.0263
North Woolwich          51.4987,   0.0462
Erith                   51.4818,   0.1868
Tilbury                 51.4567,   0.3350
Gravesend Denton        51.4445,   0.3948
Coryton Thameshaven     51.5047,   0.5052
Coryton No5 Jetty       51.5070,   0.5232
Coryton Holehaven       51.5118,   0.5512
Southend Pier           51.5145,   0.7238
Shivering Sands         51.4728,   1.1065
Margate Pile            51.3947,   1.3788
Margate Harbour         51.3920,   1.3780
Walton-on-the-Naze      51.8433,   1.2800

GAUGE_BBOX (all 16 gauges): [52.14333333333333, -0.6174999999999999, 51.092000000000006, 1.6788333333333334]
SURGE_BBOX (fetch region): [55, -3, 51, 4]
ACTIVE_GAUGES: ['Southend Pier']
Using ACTIVE_BBOX: [51.6145, 0.6238333333333334, 51.4145, 0.8238333333333333]


## Sizing the request now that the area is small

Since `ACTIVE_BBOX` is now just a small box around Southend rather than the full 16-gauge network or the wide surge region, it has roughly a quarter of `GAUGE_BBOX`'s grid points, which was itself about a ninth of `SURGE_BBOX`. That headroom means the three variables can go back to being requested together rather than split out one at a time, and the time window can be a calendar quarter instead of a single month, both of which mean far fewer requests: 21 years x 4 quarters = 84, rather than 756.

`MONTHS_PER_CHUNK` below controls the time window (3 means quarterly). If a request still comes back with "cost limits exceeded" at this size, that's the first thing to try changing: drop it to 1 for monthly and re-run. Because CDS rejects an oversized request immediately rather than after a long queue wait, as you saw last time, you'll know within the first request or two whether quarterly works, there's no real time cost to trying the larger chunk first.

As before, the loop checks for each file before requesting it and prints a running count, so it's safe to stop and resume across sessions, and nothing gets written to disk for a request that fails outright.


In [ ]:
import cdsapi
import os
import time
import random

def download_era5_chunk(year, start_month, n_months, variables, area, out_dir, client=None):
    """Download n_months of ERA5 data starting at start_month, for all given variables together."""
    end_month = start_month + n_months - 1
    out_path = os.path.join(out_dir, f"era5_{year}_{start_month:02d}-{end_month:02d}.nc")

    if client is None:
        client = cdsapi.Client()

    request = {
        "product_type": "reanalysis",
        "variable": variables,
        "year": [str(year)],
        "month": [f"{m:02d}" for m in range(start_month, end_month + 1)],
        "day": [f"{d:02d}" for d in range(1, 32)],
        "time": [f"{h:02d}:00" for h in range(24)],
        "area": area,
        "format": "netcdf",
    }

    client.retrieve("reanalysis-era5-single-levels", request, out_path)
    return out_path


In [ ]:
VARIABLES = ["mean_sea_level_pressure", "10m_u_component_of_wind", "10m_v_component_of_wind"]
YEARS = range(2020, 2025)
MONTHS_PER_CHUNK = 3  # quarterly; drop to 1 if this still trips the cost limit

chunk_starts = list(range(1, 13, MONTHS_PER_CHUNK))
tasks = [(y, m) for y in YEARS for m in chunk_starts]

def task_path(year, start_month):
    end_month = start_month + MONTHS_PER_CHUNK - 1
    return os.path.join(era5_raw_path, f"era5_{year}_{start_month:02d}-{end_month:02d}.nc")

already_done = sum(os.path.exists(task_path(y, m)) for y, m in tasks)
print(f"{len(tasks)} requests total, {already_done} already on disk, {len(tasks) - already_done} left to fetch.")

client = cdsapi.Client()
failed = []

for i, (year, start_month) in enumerate(tasks, start=1):
    if os.path.exists(task_path(year, start_month)):
        continue
    try:
        download_era5_chunk(year, start_month, MONTHS_PER_CHUNK, VARIABLES, ACTIVE_BBOX, era5_raw_path, client=client)
        print(f"[{i}/{len(tasks)}] done: {year} chunk starting month {start_month:02d}")
        time.sleep(random.uniform(15, 40))
    except Exception as e:
        print(f"[{i}/{len(tasks)}] FAILED: {year} chunk starting month {start_month:02d}: {e}")
        failed.append((year, start_month))

print()
if failed:
    print(f"{len(failed)} requests failed, re-run this cell to retry just those.")
    print("If they keep failing, set MONTHS_PER_CHUNK = 1 above and re-run.")
else:
    print("All requests completed.")


20 requests total, 10 already on disk, 10 left to fetch.


2026-07-25 20:43:41,179 INFO Request ID is 416a985a-ed5c-4174-9dba-4cf3f53890dc
INFO:ecmwf.datastores.legacy_client:Request ID is 416a985a-ed5c-4174-9dba-4cf3f53890dc
2026-07-25 20:43:41,327 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-07-25 20:46:35,305 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-07-25 20:52:03,503 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


eb05a97213df9d105f552437e0f49c91.nc:   0%|          | 0.00/207k [00:00<?, ?B/s]

[11/20] done: 2022 chunk starting month 07


2026-07-25 20:52:28,517 INFO Request ID is 51e9204d-bb82-4459-b98a-06456ce58519
INFO:ecmwf.datastores.legacy_client:Request ID is 51e9204d-bb82-4459-b98a-06456ce58519
2026-07-25 20:52:28,667 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-07-25 20:53:01,967 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-07-25 20:58:51,403 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


...

9386978f866cc0c8ae4205bfc32c2f43.nc:   0%|          | 0.00/207k [00:00<?, ?B/s]

[19/20] done: 2024 chunk starting month 07


2026-07-25 21:51:18,412 INFO Request ID is c8b72d46-d037-4d01-bb0f-95c6fe463f3f
INFO:ecmwf.datastores.legacy_client:Request ID is c8b72d46-d037-4d01-bb0f-95c6fe463f3f
2026-07-25 21:51:18,543 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-07-25 21:51:32,462 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-07-25 21:57:39,068 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


942c01cf848755bd368c4cec986b8d56.nc:   0%|          | 0.00/207k [00:00<?, ?B/s]

[20/20] done: 2024 chunk starting month 10

All requests completed.


## River discharge and catchment rainfall (NRFA)

Both come from the same station: Thames at Kingston, NRFA station 39001, just upstream of Teddington Weir (the tidal limit). No account or key is needed: this is a fully open API.

- `gdf` = gauged daily flow (m3/s)
- `cdr` = catchment daily rainfall (mm/day), spatially averaged over the catchment upstream of the gauge

The response is NRFA's own CSV format, which has a metadata header of variable length before the actual data table starts. The code below saves the raw response as-is (worth keeping, it documents the record's provenance) and separately auto-detects where the data rows begin, by looking for the first block of consecutive lines that start with a parseable date. Check the printed preview before trusting it downstream: it's a heuristic, not a guaranteed fixed column count.


In [ ]:
import requests
import pandas as pd
import io

NRFA_BASE = "https://nrfaapps.ceh.ac.uk/nrfa/ws/time-series"
STATION = 39001
STUDY_START, STUDY_END = '2004-01-01', '2024-12-31'

def fetch_nrfa(data_type, station=STATION):
    """Download an NRFA time series (data_type = 'gdf' or 'cdr') as raw CSV text."""
    params = {"format": "nrfa-csv", "data-type": data_type, "station": station}
    r = requests.get(NRFA_BASE, params=params, timeout=60)
    r.raise_for_status()
    return r.text

def parse_nrfa_csv(raw_text, min_consecutive=5):
    """Best-effort parse: locate the first run of lines whose first field is a real date, and treat that as the start of the data block."""
    lines = raw_text.splitlines()
    start_idx = None
    for i in range(len(lines) - min_consecutive):
        candidate_dates = [pd.to_datetime(lines[j].split(',')[0], errors='coerce') for j in range(i, i + min_consecutive)]
        if all(d is not pd.NaT for d in candidate_dates):
            start_idx = i
            break
    if start_idx is None:
        raise ValueError("Couldn't find a run of date-like rows, inspect the raw text manually.")

    data_lines = lines[start_idx:]
    n_cols = len(data_lines[0].split(','))
    col_names = ['date', 'value'] + [f'col_{k}' for k in range(2, n_cols)]

    try:
        joined = chr(10).join(data_lines)
        df = pd.read_csv(io.StringIO(joined), header=None, names=col_names)
    except Exception as e:
        print("Automatic parsing failed, first 10 raw data lines for manual inspection:")
        for line in data_lines[:10]:
            print(line)
        raise e

    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date']).set_index('date')
    return df

# River discharge
flow_raw = fetch_nrfa('gdf')
with open(os.path.join(nrfa_path, 'kingston_flow_raw.csv'), 'w') as f:
    f.write(flow_raw)
flow_df = parse_nrfa_csv(flow_raw).loc[STUDY_START:STUDY_END]
print("Discharge preview (study period):")
print(flow_df.head())

print()

# Catchment rainfall
rain_raw = fetch_nrfa('cdr')
with open(os.path.join(nrfa_path, 'kingston_catchment_rainfall_raw.csv'), 'w') as f:
    f.write(rain_raw)
rain_df = parse_nrfa_csv(rain_raw).loc[STUDY_START:STUDY_END]
print("Catchment rainfall preview (study period):")
print(rain_df.head())


Discharge preview (study period):
            value
date             
2004-01-01   79.2
2004-01-02   99.6
2004-01-03   81.1
2004-01-04   62.7
2004-01-05   57.6

Catchment rainfall preview (study period):
            value
date             
2004-01-01    1.5
2004-01-02    0.0
2004-01-03    0.6
2004-01-04    0.5
2004-01-05    2.7


## Consolidating to one file per variable group

Files in `era5_raw/` are now one per (year, quarter) chunk, each holding all three variables together, so there's a single consolidation function again rather than one per variable. It opens everything lazily with `xarray.open_mfdataset`, extracts the nearest ERA5 grid cell to each gauge, and writes two wide CSVs:

- `pressure_all_gauges.csv`
- `wind_all_gauges.csv` (u and v components, one pair of columns per gauge)

One thing worth being deliberate about: this only extracts gauges listed in `ACTIVE_GAUGES_DICT`, not the full `GAUGES` dict. `xarray`'s nearest-neighbour lookup doesn't fail or warn when a point falls outside the downloaded box, it just silently returns whatever grid cell happens to be closest, even if that's the edge of the box and nowhere near the actual gauge. Since `ACTIVE_BBOX` only covers Southend right now, running this against the full gauge list would quietly produce meaningless values for the other thirteen rather than an error telling you something's wrong.


In [ ]:
import os
import glob
import pandas as pd
import xarray as xr

def consolidate_to_gauges(nc_dir, gauges, variables, out_csv):
    """Open every ERA5 chunk file lazily, extract the nearest grid cell to each gauge, and write one wide CSV."""
    files = sorted(glob.glob(os.path.join(nc_dir, "era5_*.nc")))
    if not files:
        raise FileNotFoundError(f"No ERA5 files found in {nc_dir}")

    ds = xr.open_mfdataset(files, combine='by_coords')
    available = list(ds.data_vars)
    missing = [v for v in variables if v not in available]
    if missing:
        print(f"Warning: {missing} not found. Variables in file: {available}")

    frames = {}
    for name, (lat, lon) in gauges.items():
        point = ds.sel(latitude=lat, longitude=lon, method='nearest')
        for var in variables:
            if var in point:
                frames[f"{name}_{var}"] = point[var].to_pandas()

    wide = pd.DataFrame(frames)
    wide.to_csv(out_csv)
    ds.close()
    print(f"Saved {wide.shape[0]} rows x {wide.shape[1]} columns to {out_csv}")
    return wide

pressure_csv = os.path.join(consolidated_path, 'pressure_all_gauges.csv')
wind_csv = os.path.join(consolidated_path, 'wind_all_gauges.csv')

pressure_df = consolidate_to_gauges(era5_raw_path, ACTIVE_GAUGES_DICT, ['msl'], pressure_csv)
wind_df = consolidate_to_gauges(era5_raw_path, ACTIVE_GAUGES_DICT, ['u10', 'v10'], wind_csv)


Saved 184104 rows x 1 columns to /content/drive/MyDrive/tidal_analysis_and_prediction/meteo_data/consolidated/pressure_all_gauges.csv
Saved 184104 rows x 2 columns to /content/drive/MyDrive/tidal_analysis_and_prediction/meteo_data/consolidated/wind_all_gauges.csv


In [ ]:
flow_csv = os.path.join(consolidated_path, 'river_discharge_kingston.csv')
rain_csv = os.path.join(consolidated_path, 'rainfall_kingston_catchment.csv')

flow_df.to_csv(flow_csv)
rain_df.to_csv(rain_csv)

print("Consolidated files:")
for path in [pressure_csv, wind_csv, flow_csv, rain_csv]:
    size_kb = os.path.getsize(path) / 1024
    print(f"  {path}  ({size_kb:.0f} KB)")

print()
print(f"Pressure/wind date range: {pressure_df.index.min()} to {pressure_df.index.max()}")
print(f"Discharge/rainfall date range: {flow_df.index.min()} to {flow_df.index.max()}")


Consolidated files:
  /content/drive/MyDrive/tidal_analysis_and_prediction/meteo_data/consolidated/pressure_all_gauges.csv  (5400 KB)
  /content/drive/MyDrive/tidal_analysis_and_prediction/meteo_data/consolidated/wind_all_gauges.csv  (7257 KB)
  /content/drive/MyDrive/tidal_analysis_and_prediction/meteo_data/consolidated/river_discharge_kingston.csv  (121 KB)
  /content/drive/MyDrive/tidal_analysis_and_prediction/meteo_data/consolidated/rainfall_kingston_catchment.csv  (107 KB)

Pressure/wind date range: 2004-01-01 00:00:00 to 2024-12-31 23:00:00
Discharge/rainfall date range: 2004-01-01 00:00:00 to 2024-09-30 00:00:00


## Next steps

The four consolidated CSVs in `meteo_data/consolidated/` are ready to resample and join onto your 10-minute Southend observations, using the same upsample-and-merge approach as your local pipeline (linear interpolation for pressure and wind, forward-fill for the daily discharge and rainfall).

This notebook deliberately stops at NRFA's catchment-averaged rainfall rather than also pulling hourly point rainfall from CEDA's MIDAS Open archive. That's a reasonable place to stop for now: CEDA needs a separate registration and its own credential flow, and the catchment series is the more physically meaningful driver of discharge anyway. Worth adding later only if you decide you specifically need sub-daily, point-level rainfall near the estuary rather than the upstream catchment average.
